In [ ]:
import os
from typing import Optional, List
import pandas as pd

# Download data from Kaggle and place in a directory, then use the function below to load and filter for 2022 rows.
# https://www.kaggle.com/datasets/frankcaoyun/stocktwits-2020-2022-raw

# Loading all CSV files in a directory (and subdirectories), filtering for rows where the `created_at` column is in the year 2022, and 
# optionally saving the combined results to a new CSV file. 
# This function is memory-efficient by processing files in chunks and streaming results to disk if a save path is provided.
def load_year_data_from_dir(root_dir: str, created_at_col: str = 'created_at', chunksize: int = 100_000, year: int = 2022, save_path: Optional[str] = None) -> pd.DataFrame:
    """Recursively read all .csv files under `root_dir`, keep only rows where `created_at` is in the specified year.

    Parameters:
    - root_dir: path to directory containing csv files (recursed).
    - created_at_col: name of the timestamp column in the csvs.
    - chunksize: rows per chunk when reading each csv (memory-safe).
    - year: the year to filter rows for.
    - save_path: optional file path to write the combined rows for the specified year as a csv. If provided, rows are streamed to that file to avoid keeping everything in memory.

    Returns:
    - pd.DataFrame containing all rows from the specified year (unless `save_path` used, in which case an empty DataFrame is returned and the combined CSV is written).
    """
    csv_files: List[str] = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if fn.lower().endswith('.csv'):
                csv_files.append(os.path.join(dirpath, fn))
    if not csv_files:
        raise FileNotFoundError(f'No CSV files found under {root_dir}')

    # helper to filter a chunk for the requested year
    def _filter_chunk(df: pd.DataFrame) -> pd.DataFrame:
        if created_at_col not in df.columns:
            return pd.DataFrame(columns=df.columns)
        # parse to datetime (coerce errors) then filter by requested year
        df[created_at_col] = pd.to_datetime(df[created_at_col], errors='coerce', utc=True)
        return df[df[created_at_col].dt.year == year]

    collected: List[pd.DataFrame] = []
    wrote_header = False
    if save_path:
        # ensure directory exists
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)

    for fpath in csv_files:
        try:
            for chunk in pd.read_csv(fpath, chunksize=chunksize, dtype=str, keep_default_na=False):
                # convert only the relevant column to datetime inside filter
                filtered = _filter_chunk(chunk)
                if filtered.empty:
                    continue
                if save_path:
                    # stream to disk to avoid memory pressure
                    filtered.to_csv(save_path, mode='a', header=not wrote_header, index=False)
                    wrote_header = True
                else:
                    collected.append(filtered)
        except pd.errors.EmptyDataError:
            continue
        except Exception as e:
            # continue on individual file errors but notify user by printing a warning
            print(f'Warning: failed to read {fpath}: {e}')
            continue

    if save_path:
        # when saved to disk, return empty DataFrame (file contains combined results)
        return pd.DataFrame()

    if not collected:
        return pd.DataFrame()

    # combine and ensure created_at is datetime and set UTC tz if present
    combined = pd.concat(collected, ignore_index=True)
    if created_at_col in combined.columns:
        combined[created_at_col] = pd.to_datetime(combined[created_at_col], errors='coerce', utc=True)
    return combined

In [16]:
# Example usage (adjust `root` as needed):
root = '../data/StockTwits_2020_2022_Raw/AAPL_2020_2022'
# load into memory (can be large)
# df_2022 = load_2022_from_dir(root)

# stream to a single csv on disk (recommended for large datasets):
out_csv = '../data/Sentiment_Year_Data/AAPL_2022_only.csv'
load_year_data_from_dir(root, year=2022, save_path=out_csv)

# When using `save_path`, open that file with pandas when you want to work with it:
# df_2022 = pd.read_csv(out_csv, parse_dates=['created_at'])

print('Utility `load_year_data_from_dir` defined. Use as shown in comments.')

Utility `load_year_data_from_dir` defined. Use as shown in comments.


In [11]:
# Example usage (adjust `root` as needed):
root = '../data/StockTwits_2020_2022_Raw/AMZN2019-2022'
# load into memory (can be large)
# df_2022 = load_2022_from_dir(root)

# stream to a single csv on disk (recommended for large datasets):
out_csv = '../data/Sentiment_Year_Data/AMZN_2022_only.csv'
load_year_data_from_dir(root, year=2022, save_path=out_csv)

# When using `save_path`, open that file with pandas when you want to work with it:
# df_2022 = pd.read_csv(out_csv, parse_dates=['created_at'])

print('Utility `load_year_data_from_dir` defined. Use as shown in comments.')

Utility `load_year_data_from_dir` defined. Use as shown in comments.


In [12]:
# Example usage (adjust `root` as needed):
root = '../data/StockTwits_2020_2022_Raw/TSLA_2020_2022'
# load into memory (can be large)
# df_2022 = load_2022_from_dir(root)

# stream to a single csv on disk (recommended for large datasets):
out_csv = '../data/Sentiment_Year_Data/TSLA_2022_only.csv'
load_year_data_from_dir(root, year=2022, save_path=out_csv)

# When using `save_path`, open that file with pandas when you want to work with it:
# df_2022 = pd.read_csv(out_csv, parse_dates=['created_at'])

print('Utility `load_year_data_from_dir` defined. Use as shown in comments.')

Utility `load_year_data_from_dir` defined. Use as shown in comments.
